# 4 - HCP background dependence dataframe

This notebook is computes, for every subject, stimulated region, and time point, the baseline/evoked energy measures and perturbation effects defined in the background-dependence notes.

The notebook assumes that:

1. `HCP_1_Process_data.ipynb` has already been run and the processed arrays are available.
2. `HCP_2_Fit_data.ipynb` has already been run and trained models are available.
3. `HCP_3_Connectiviy.ipynb` has already been run and `EC_t` have already been generated and saved per subject.

For each selected subject, the notebook:

1. Locates processed inputs, trained models, and `EC_t` files
2. Loads the subject-specific ANN model
3. Predicts the unperturbed next state `X_(t+1)`
4. Reconstructs the perturbed next state `X^(j)_(t+1)` for each stimulated ROI `j`
5. Computes baseline energy, evoked energy, effect size, and effect direction
6. Builds a long-format dataframe with one row per `(sub_id, roi, time)`
7. Saves the dataframe as CSV and PKL

## Saved outputs

The notebook saves:

- `Results/HCP_4_df_background_dependence.csv`
- `Results/HCP_4_df_background_dependence.pkl`


In [1]:
from pathlib import Path
import sys
import os
import gc
import numpy as np
import pandas as pd
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.serialization

# -----------------------------------------------------------------------------
# Path and environment configuration
# -----------------------------------------------------------------------------
# Expected repository structure:
#
# BrainStim_ANN_fMRI_HCP/
# ├── src/
# ├── notebooks/
# └── Results/
#     ├── processed/
#     ├── ANN_model/
#     └── ECts_MLP/

repo_dir = Path.cwd().resolve().parent
results_dir = repo_dir / "Results"
preproc_dir = results_dir / "processed"
models_dir = results_dir / "ANN_model"
ects_dir = results_dir / "ECts"
save_dir = results_dir 

sys.path.insert(0, str(repo_dir))

from src import NPI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repository directory :", repo_dir)
print("Results directory    :", results_dir)
print("Processed data dir   :", preproc_dir)
print("Models directory     :", models_dir)
print("EC_t directory       :", ects_dir)
print("Output directory     :", save_dir)
print("Device               :", device)


Repository directory : C:\Users\tomas\Documents\PhD\Projects\BrainStim
Results directory    : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results
Processed data dir   : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\processed
Models directory     : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\ANN_model
EC_t directory       : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\BECts
Output directory     : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results
Device               : cpu


In [2]:
# =============================================================================
# Analysis parameters
# =============================================================================
method = "MLP"
ROI_num = 450
using_steps = 3

output_csv = save_dir / "HCP_4_df_background_dependence_ECts.csv"
output_pkl = save_dir / "HCP_4_df_background_dependence_ECts.pkl"

print("Analysis configuration:")
print("  Method               :", method)
print("  Number of regions    :", ROI_num)
print("  Window length        :", using_steps)
print("  Output CSV           :", output_csv)
print("  Output PKL           :", output_pkl)


Analysis configuration:
  Method               : MLP
  Number of regions    : 450
  Window length        : 3
  Output CSV           : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\HCP_4_df_background_dependence_BECts.csv
  Output PKL           : C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\HCP_4_df_background_dependence_BECts.pkl


In [4]:
# =============================================================================
# Locate subjects with complete background-dependence inputs
# =============================================================================
input_files = sorted(preproc_dir.glob("*_inputs.npy"))
model_files = sorted(models_dir.glob(f"*_{method}.pt"))
ect_files   = sorted(ects_dir.glob("*_ECt.npy"))

subject_ids_inputs = {p.name.split("_inputs.npy")[0] for p in input_files}
subject_ids_models = {p.name.split(f"_{method}.pt")[0] for p in model_files}
subject_ids_ects   = {p.name.split("_ECt.npy")[0] for p in ect_files}

subject_ids = sorted(subject_ids_inputs & subject_ids_models & subject_ids_ects)

print(f"Subjects with complete inputs/models/EC_t: {len(subject_ids)}")
if len(subject_ids) > 0:
    print("First subjects:", subject_ids[:10])
else:
    print("No complete subjects found. Check preproc_dir, models_dir, and ects_dir.")


Subjects with complete inputs/models/EC_t: 100
First subjects: ['id_100206', 'id_100307', 'id_100408', 'id_101006', 'id_101107', 'id_101309', 'id_101915', 'id_102008', 'id_102109', 'id_102311']


## Model loading and background-dependence utilities

The following cells:
- safely load trained models saved in different PyTorch formats
- recover the baseline state `X_t` from the preprocessed ANN input windows
- compute the long-format dataframe requested for the background-dependence analysis


In [5]:
# Allowlist model classes for recent PyTorch versions
torch.serialization.add_safe_globals(
    [NPI.ANN_MLP, NPI.ANN_CNN, NPI.ANN_RNN, NPI.ANN_VAR])

def load_model(model_path, inputs=None, targets=None):
    """Load either a full serialized model or a state-dict checkpoint."""
    ckpt = torch.load(model_path, map_location=device, weights_only=False)

    if hasattr(ckpt, "eval"):
        model = ckpt.to(device)
        model.eval()
        return model

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        method_ckpt = ckpt.get("method", method)
        roi_ckpt = ckpt.get("ROI_num", targets.shape[-1] if targets is not None else ROI_num)
        steps_ckpt = ckpt.get("using_steps", using_steps)
        model = NPI.build_model(method_ckpt, roi_ckpt, steps_ckpt).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        return model

    raise ValueError("Unrecognized model format.")


def recover_current_state(inputs, n_regions):
    """
    Recover X_t from the last block of each flattened ANN input window.

    Expected input shape after preprocessing:
    (n_time, using_steps * n_regions)

    The final n_regions entries correspond to the current state at time t.
    """
    inputs = np.asarray(inputs)

    if inputs.ndim != 2:
        raise ValueError(f"Expected inputs to be 2D, got shape {inputs.shape}")

    if inputs.shape[1] % n_regions != 0:
        raise ValueError(
            f"Input width {inputs.shape[1]} is not divisible by n_regions={n_regions}"
        )

    return inputs[:, -n_regions:]


def build_background_dependence_df_for_subject(sid, model, inputs, EC_t):
    """
    Build one long-format dataframe for a single subject.

    Parameters
    ----------
    sid : str
        Subject identifier.
    model : torch.nn.Module
        Trained subject-specific ANN model.
    inputs : ndarray, shape (T, S*N)
        Preprocessed ANN input windows.
    EC_t : ndarray, shape (T, N, N)
        Virtual perturbation tensor where:
        EC_t[t, j, i] = X^(j)_(t+1, i) - X_(t+1, i)

    Returns
    -------
    df_sub : pandas.DataFrame
        Long-format dataframe with one row per (subject, roi, time).
    """
    T_eff, N, N_check = EC_t.shape
    if N != N_check:
        raise ValueError(f"EC_t must be square in its last two dims, got {EC_t.shape}")

    # ------------------------------------------------------------------
    # Recover baseline state X_t
    # ------------------------------------------------------------------
    X_t = recover_current_state(inputs[:T_eff], N)   # shape: (T, N)
    
    # ------------------------------------------------------------------
    # Predict unperturbed next state X_{t+1}
    # ------------------------------------------------------------------
    with torch.no_grad():
        X_tp1 = model(torch.tensor(inputs[:T_eff], dtype=torch.float32, device=device))
        X_tp1 = X_tp1.detach().cpu().numpy()            # shape: (T, N)

    # ------------------------------------------------------------------
    # Build evoked next state X^(j)_{t+1}
    # ------------------------------------------------------------------
    X_evoked = X_tp1[:, None, :] + EC_t                 # shape: (T, N_target, N)

    # ------------------------------------------------------------------
    # Baseline energies
    # ------------------------------------------------------------------
    global_baseline_energy = np.sum(X_t ** 2, axis=1)   # shape: (T,)
    local_baseline_energy = X_t ** 2                    # shape: (T, N)

    # ------------------------------------------------------------------
    # Global measures
    # ------------------------------------------------------------------
    global_evoked_energy = np.sum(X_evoked ** 2, axis=2)   # shape: (T, N)
    global_effect_size = np.sum(EC_t ** 2, axis=2)         # shape: (T, N)
    global_effect_direction = np.sum(EC_t, axis=2)         # shape: (T, N)

    # ------------------------------------------------------------------
    # Local measures (diagonal: stimulate j, measure j)
    # ------------------------------------------------------------------
    roi_idx = np.arange(N, dtype=int)
    local_evoked_state = X_evoked[:, roi_idx, roi_idx]     # shape: (T, N)
    local_effect_state = EC_t[:, roi_idx, roi_idx]         # shape: (T, N)

    
    local_evoked_energy = local_evoked_state ** 2          # shape: (T, N)
    local_effect_size = local_effect_state ** 2            # shape: (T, N)
    local_effect_direction = local_effect_state            # shape: (T, N)

    # ------------------------------------------------------------------
    # Long-format indexing
    # ------------------------------------------------------------------
    time_idx = np.arange(T_eff, dtype=int)

    sub_id_col = np.repeat(sid, T_eff * N)
    roi_col = np.tile(roi_idx, T_eff)
    time_col = np.repeat(time_idx, N)

    global_baseline_energy_col = np.repeat(global_baseline_energy, N)
    global_evoked_energy_col = global_evoked_energy.reshape(-1)
    global_effect_size_col = global_effect_size.reshape(-1)
    global_effect_direction_col = global_effect_direction.reshape(-1)

    local_baseline_energy_col = local_baseline_energy.reshape(-1)
    local_evoked_energy_col = local_evoked_energy.reshape(-1)
    local_effect_size_col = local_effect_size.reshape(-1)
    local_effect_direction_col = local_effect_direction.reshape(-1)

    # ------------------------------------------------------------------
    # Dataframe
    # ------------------------------------------------------------------
    df_sub = pd.DataFrame({
        "sub_id": sub_id_col,
        "roi": roi_col,
        "time": time_col,
        "global_baseline_energy": global_baseline_energy_col,
        "global_evoked_energy": global_evoked_energy_col,
        "global_effect_size": global_effect_size_col,
        "global_effect_direction": global_effect_direction_col,
        "local_baseline_energy": local_baseline_energy_col,
        "local_evoked_energy": local_evoked_energy_col,
        "local_effect_size": local_effect_size_col,
        "local_effect_direction": local_effect_direction_col,
    })

    return df_sub

In [6]:
# =============================================================================
# Build the full background-dependence dataframe
# =============================================================================
all_subject_dfs = []
failed_subjects = []

for sid in subject_ids:
    print(f"Processing subject {sid}")

    input_path = preproc_dir / f"{sid}_inputs.npy"
    model_path = models_dir / f"{sid}_{method}.pt"
    ect_path   = ects_dir / f"{sid}_ECt.npy"

    try:
        inputs = np.load(input_path)
        EC_t = np.load(ect_path)
        model = load_model(model_path, inputs=inputs)

        df_sub = build_background_dependence_df_for_subject(
            sid=sid,
            model=model,
            inputs=inputs,
            EC_t=EC_t,
        )

        all_subject_dfs.append(df_sub)
        #print(f"  inputs shape  : {inputs.shape}")
        #print(f"  EC_t shape    : {EC_t.shape}")
        #print(f"  rows added    : {len(df_sub):,}")

        del model, inputs, EC_t, df_sub
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        failed_subjects.append((sid, str(exc)))
        print(f"  Failed: {exc}")

if len(all_subject_dfs) == 0:
    raise RuntimeError("No subject dataframe could be created.")

HCP_4_df_background_dependence = pd.concat(all_subject_dfs, axis=0, ignore_index=True)

print("Final dataframe built successfully.")
print("Shape:", HCP_4_df_background_dependence.shape)
print("Columns:")
print(list(HCP_4_df_background_dependence.columns))

if len(failed_subjects) > 0:
    print("Subjects that failed during processing:")
    for sid, msg in failed_subjects:
        print(f"  {sid}: {msg}")


Processing subject id_100206
Processing subject id_100307
Processing subject id_100408
Processing subject id_101006
Processing subject id_101107
Processing subject id_101309
Processing subject id_101915
Processing subject id_102008
Processing subject id_102109
Processing subject id_102311
Processing subject id_102513
Processing subject id_102614
Processing subject id_102715
Processing subject id_102816
Processing subject id_103010
Processing subject id_103111
Processing subject id_103212
Processing subject id_103414
Processing subject id_103515
Processing subject id_103818
Processing subject id_104012
Processing subject id_104416
Processing subject id_104820
Processing subject id_105014
Processing subject id_105115
Processing subject id_105216
Processing subject id_105620
Processing subject id_105923
Processing subject id_106319
Processing subject id_106521
Processing subject id_106824
Processing subject id_107018
Processing subject id_107321
Processing subject id_107422
Processing sub

In [7]:
# =============================================================================
# Quick checks
# =============================================================================
print(HCP_4_df_background_dependence.head())
print()
print(HCP_4_df_background_dependence.describe(include="all").transpose().head(15))


      sub_id  roi  time  global_baseline_energy  global_evoked_energy  \
0  id_100206    0     0              231.013324            300.487568   
1  id_100206    1     0              231.013324            300.197023   
2  id_100206    2     0              231.013324            300.699545   
3  id_100206    3     0              231.013324            301.268095   
4  id_100206    4     0              231.013324            301.148623   

   global_effect_size  global_effect_direction  local_baseline_energy  \
0            0.281620                11.190036               0.101116   
1            0.257475                10.691392               0.132412   
2            0.330784                12.143571               0.008565   
3            0.415552                13.630528               1.187434   
4            0.433332                13.920187               0.105217   

   local_evoked_energy  local_effect_size  local_effect_direction  
0             0.114951           0.000613             

In [8]:
# =============================================================================
# Save dataframe
# =============================================================================
HCP_4_df_background_dependence.to_csv(output_csv, index=False)
HCP_4_df_background_dependence.to_pickle(output_pkl)

print("Saved:")
print("  ", output_csv)
print("  ", output_pkl)

Saved:
   C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\HCP_4_df_background_dependence_BECts.csv
   C:\Users\tomas\Documents\PhD\Projects\BrainStim\Results\HCP_4_df_background_dependence_BECts.pkl
